In [1]:
import sys
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras as keras
sys.modules["tensorflow.keras"] = keras
sys.modules["keras"] = keras

import tensorflow as tf

from tf_keras.models import Sequential
from tf_keras.layers import MaxPooling2D, Activation, Flatten, Dropout, BatchNormalization

2026-04-29 00:53:47.002744: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777416827.126130    1381 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777416827.159936    1381 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-29 00:53:47.460979: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import scipy.io
import numpy as np

def load_svhn(path):
    data = scipy.io.loadmat(path)
    X = data["X"]
    y = data["y"].reshape(-1)

    X = np.transpose(X, (3,0,1,2))

    y[y == 10] = 0

    return X, y

In [3]:
import gc

X_train, y_train = load_svhn("/home/slopin/DAT255-project/SVHN/Data/train_32x32.mat")
X_test, y_test   = load_svhn("/home/slopin/DAT255-project/SVHN/Data/test_32x32.mat")

#Adding the extra data, but only a subset of it to avoid memory issues.
X_ex_tmp, y_ex_tmp = load_svhn("/home/slopin/DAT255-project/SVHN/Data/extra_32x32.mat")
indices = np.random.choice(len(X_ex_tmp), 100000, replace=False)
X_extra_sampled = X_ex_tmp[indices]
y_extra_sampled = y_ex_tmp[indices]

#Garbage collecting to free up memory.
del X_ex_tmp, y_ex_tmp
gc.collect()

#Concatenating the extra data with the training data.
X_train = np.concatenate((X_train, X_extra_sampled), axis=0)
y_train = np.concatenate((y_train, y_extra_sampled), axis=0)

X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

y_train = keras.utils.to_categorical(y_train, 10)
y_test  = keras.utils.to_categorical(y_test, 10)

input_shape = (32, 32, 3)

In [5]:
data_augmentation = keras.Sequential([
    keras.layers.RandomTranslation(height_factor=0.02, width_factor=0.02),
    keras.layers.RandomZoom(0.05),
    keras.layers.RandomContrast(0.02),
])

inputs = keras.Input(shape=input_shape)

x = data_augmentation(inputs)

x = keras.layers.Conv2D(32, (3,3), padding="same")(x)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = MaxPooling2D()(x)

x = keras.layers.Conv2D(64, (3,3), padding="same")(x)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = MaxPooling2D()(x)

x = keras.layers.Conv2D(128, (3,3), padding="same")(x)
x = BatchNormalization()(x)
x = Activation("relu")(x)
x = MaxPooling2D()(x)

x = Flatten()(x)

x = keras.layers.Dense(512)(x)
x = Activation("relu")(x)

x = Dropout(0.2)(x)
outputs = keras.layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs=inputs, outputs=outputs)

In [6]:
opt = keras.optimizers.Adam(learning_rate=1e-3, amsgrad=True)
model.compile(
    optimizer=opt,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.2, patience=4, min_lr=1e-7, verbose=1),
    keras.callbacks.CSVLogger("SVHN_baseline_log.csv", append=True)
]

In [7]:
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=60,
    batch_size=256,
    callbacks=callbacks,
    shuffle=True
)


2026-04-29 00:54:20.846701: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2128982016 exceeds 10% of free system memory.
2026-04-29 00:54:23.712127: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2128982016 exceeds 10% of free system memory.


Epoch 1/60


I0000 00:00:1777416867.437402    1567 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1777416869.096746    1568 service.cc:148] XLA service 0x7847244bd170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777416869.097323    1568 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2026-04-29 00:54:29.118315: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777416869.263474    1568 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


677/677 [==============================] - 18s 20ms/step - loss: 0.9267 - accuracy: 0.7089 - val_loss: 0.5069 - val_accuracy: 0.8456 - lr: 0.0010
Epoch 2/60
677/677 [==============================] - 12s 17ms/step - loss: 0.3003 - accuracy: 0.9117 - val_loss: 0.3618 - val_accuracy: 0.8946 - lr: 0.0010
Epoch 3/60
677/677 [==============================] - 11s 16ms/step - loss: 0.2428 - accuracy: 0.9283 - val_loss: 0.3142 - val_accuracy: 0.9115 - lr: 0.0010
Epoch 4/60
677/677 [==============================] - 12s 18ms/step - loss: 0.2108 - accuracy: 0.9389 - val_loss: 0.2864 - val_accuracy: 0.9208 - lr: 0.0010
Epoch 5/60
677/677 [==============================] - 12s 18ms/step - loss: 0.1911 - accuracy: 0.9445 - val_loss: 0.3343 - val_accuracy: 0.9019 - lr: 0.0010
Epoch 6/60
677/677 [==============================] - 11s 16ms/step - loss: 0.1746 - accuracy: 0.9498 - val_loss: 0.2652 - val_accuracy: 0.9270 - lr: 0.0010
Epoch 7/60
677/677 [==============================] - 12s 18ms/step -

In [8]:
score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.19746990501880646
Test accuracy: 0.9516748785972595


In [9]:
model.save('Model_baseline.h5')

/home/slopin/DAT255-project/.venv_qkeras2/lib/python3.10/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
